# Parcial 2

Clasificación del rango de precio de celulares (`price_range`).


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)
df.head()


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,px_height,px_width,ram,talk_time,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,20,756,2549,19,0,1,Mid_Low
1,1021,1,0.5,1,0,1,53,0.7,136,3,905,1988,2631,7,1,0,Mid
2,563,1,0.5,1,2,1,41,0.9,145,5,1263,1716,2603,9,1,0,Mid
3,615,1,2.5,0,0,0,10,0.8,131,6,1216,1786,2769,11,0,0,Mid
4,1821,1,1.2,0,13,1,44,0.6,141,2,1208,1212,1411,15,1,0,Mid_Low


## Cantidad de observaciones por `price_range`


In [2]:
import plotly.express as px

orden = ['Low', 'Mid_Low', 'Mid', 'High']
conteo = (
    df['price_range']
    .value_counts()
    .reindex(orden)
    .rename_axis('price_range')
    .reset_index(name='cantidad')
)

fig = px.bar(
    conteo,
    x='price_range',
    y='cantidad',
    text='cantidad',
    title='Cantidad de observaciones por price_range'
)
fig.update_layout(xaxis={'categoryorder': 'array', 'categoryarray': orden})
fig.show()


## RFECV

División 80/20 sin `stratify`, escalado solo con entrenamiento y regresión logística (`random_state=0`, `max_iter=500`) con `RFECV(cv=5)`.


In [3]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns='price_range')
y = df['price_range']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

escalador = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X.columns, index=X_test.index)

modelo_rfe = LogisticRegression(random_state=0, max_iter=500)
rfecv = RFECV(estimator=modelo_rfe, cv=5)
rfecv.fit(X_train_s, y_train)

print('Número óptimo de variables:', rfecv.n_features_)
print('Variables seleccionadas:', list(X.columns[rfecv.support_]))


Número óptimo de variables: 5
Variables seleccionadas: ['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram']


## Multicolinealidad

Hay multicolinealidad si alguna correlación entre las variables seleccionadas es mayor a 0.8 en valor absoluto.


In [4]:
seleccionadas = list(X.columns[rfecv.support_])
corr = X_train[seleccionadas].corr().round(4)
print(corr)

mayor = corr.abs().where(~corr.eq(1)).max().max()
print('Correlación absoluta más alta:', round(float(mayor), 4))
print('¿Hay multicolinealidad?', bool(mayor > 0.8))

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlación de las variables seleccionadas'
)
fig.show()


               battery_power  mobile_wt  px_height  px_width     ram
battery_power         1.0000     0.0049     0.0176   -0.0017  0.0245
mobile_wt             0.0049     1.0000     0.0314   -0.0003  0.0086
px_height             0.0176     0.0314     1.0000    0.4947 -0.0392
px_width             -0.0017    -0.0003     0.4947    1.0000 -0.0014
ram                   0.0245     0.0086    -0.0392   -0.0014  1.0000
Correlación absoluta más alta: 0.4947
¿Hay multicolinealidad? False


## Desempeño con las variables seleccionadas


In [5]:
from sklearn.metrics import accuracy_score

X_train_sel = X_train_s[seleccionadas]
X_test_sel = X_test_s[seleccionadas]

modelo = LogisticRegression(random_state=0)
modelo.fit(X_train_sel, y_train)

accuracy_train = accuracy_score(y_train, modelo.predict(X_train_sel))
accuracy_test = accuracy_score(y_test, modelo.predict(X_test_sel))
diferencia = accuracy_train - accuracy_test

print(f'Accuracy_train = {accuracy_train:.4f}')
print(f'Accuracy_test = {accuracy_test:.4f}')
print(f'Diferencia = {diferencia:.4f}')


Accuracy_train = 0.9723
Accuracy_test = 0.9676
Diferencia = 0.0047


## Matriz de confusión


In [6]:
from sklearn.metrics import confusion_matrix

clases = ['Low', 'Mid_Low', 'Mid', 'High']
y_pred = modelo.predict(X_test_sel)

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=clases),
    index=[f'Real: {c}' for c in clases],
    columns=[f'Predicho: {c}' for c in clases]
)

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de confusión'
)
fig.show()
cm


,Predicho: Low,Predicho: Mid_Low,Predicho: Mid,Predicho: High
Real: Low,93,0,0,0
Real: Mid_Low,3,76,1,0
Real: Mid,0,3,85,4
Real: High,0,0,1,104
